# Employee Attrition Prediction – EDA & Model Training

This notebook covers:
1. Data loading & exploration
2. Visualisations
3. Preprocessing
4. Training multiple models
5. Evaluation & model selection
6. Saving the best pipeline

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
)
import joblib
import json

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

DATA_PATH = Path('../data/Employee.csv')
MODEL_DIR = Path('../backend/model')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load & Explore

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nClass distribution:')
print(df['LeaveOrNot'].value_counts(normalize=True))
df.head()

## 2. Visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Attrition by City
pd.crosstab(df['City'], df['LeaveOrNot'], normalize='index').plot(
    kind='bar', ax=axes[0,0], color=['#34d399', '#f87171']
)
axes[0,0].set_title('Attrition Rate by City')
axes[0,0].set_ylabel('Proportion')
axes[0,0].legend(['Stay', 'Leave'])

# Attrition by Gender
pd.crosstab(df['Gender'], df['LeaveOrNot'], normalize='index').plot(
    kind='bar', ax=axes[0,1], color=['#34d399', '#f87171']
)
axes[0,1].set_title('Attrition Rate by Gender')
axes[0,1].legend(['Stay', 'Leave'])

# Attrition by Education
pd.crosstab(df['Education'], df['LeaveOrNot'], normalize='index').plot(
    kind='bar', ax=axes[1,0], color=['#34d399', '#f87171']
)
axes[1,0].set_title('Attrition Rate by Education')
axes[1,0].legend(['Stay', 'Leave'])

# Age distribution
sns.histplot(data=df, x='Age', hue='LeaveOrNot', kde=True, ax=axes[1,1], palette=['#34d399', '#f87171'])
axes[1,1].set_title('Age Distribution by Attrition')

plt.tight_layout()
plt.show()

## 3. Preprocessing & Split

In [ ]:
X = df.drop(columns=['LeaveOrNot'])
y = df['LeaveOrNot']

categorical = ['Education', 'City', 'Gender', 'EverBenched']
numerical = ['JoiningYear', 'PaymentTier', 'Age', 'ExperienceInCurrentDomain']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)
print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical),
])

## 4. Train Models

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1),
}
if HAS_XGB:
    scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    models['XGBoost'] = XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        scale_pos_weight=scale_pos, random_state=42,
        use_label_encoder=False, eval_metric='logloss', n_jobs=-1
    )

results = {}
best_name, best_auc, best_pipe = None, -1, None

for name, clf in models.items():
    print(f'\n=== {name} ===')
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_prob),
        'confusion_matrix': confusion_matrix(y_test, y_pred).tolist(),
    }
    results[name] = metrics
    for k, v in metrics.items():
        if k != 'confusion_matrix':
            print(f'  {k}: {v:.4f}')
    print('  CM:', metrics['confusion_matrix'])
    if metrics['roc_auc'] > best_auc:
        best_auc = metrics['roc_auc']
        best_name = name
        best_pipe = pipe

print(f'\n*** Best: {best_name} (AUC={best_auc:.4f}) ***')

## 5. Save Best Pipeline

In [ ]:
joblib.dump(best_pipe, MODEL_DIR / 'pipeline.pkl')

feature_importance = {}
if best_name in ('RandomForest', 'XGBoost'):
    ohe = best_pipe.named_steps['preprocessor'].named_transformers_['cat']
    cat_feats = list(ohe.get_feature_names_out(categorical))
    all_feats = numerical + cat_feats
    imps = best_pipe.named_steps['classifier'].feature_importances_
    feature_importance = dict(sorted(zip(all_feats, map(float, imps)), key=lambda x: -x[1]))

metrics_out = {
    'best_model': best_name,
    'metrics': {k: (float(v) if not isinstance(v, list) else v) for k, v in results[best_name].items()},
    'all_models': {n: {k: (float(v) if not isinstance(v, list) else v) for k, v in m.items()} for n, m in results.items()},
    'feature_importance': feature_importance,
    'features': {'categorical': categorical, 'numerical': numerical},
}
with open(MODEL_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)

print('Saved pipeline.pkl and metrics.json')